In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:

# !pip install -q pandas numpy matplotlib seaborn textblob nltk yfinance pandas_ta scikit-learn



In [ ]:
# from google.colab import files
# uploaded = files.upload()


In [ ]:
!ls /content/drive/MyDrive


In [ ]:
!ls /content/drive/MyDrive/Data-20251119T090255Z-1-001-week1


In [ ]:
!ls /content/drive/MyDrive/Data-20251119T090255Z-1-001-week1/Data


In [ ]:
!ls /content/drive/MyDrive/Data-20251119T090255Z-1-001-week1/Data/newsData.zip


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/Data-20251119T090255Z-1-001-week1/Data/newsData.zip"
extract_to = "/content/data"   # this is where the files will be extracted

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Extraction done!")


In [ ]:
!ls /content/data


In [ ]:
DATA_PATH = "/content/data/raw_analyst_ratings.csv"
import pandas as pd

df = pd.read_csv(DATA_PATH, parse_dates=['date'])
print("Shape:", df.shape)
df.head()


In [ ]:
# Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)

# Convert date safely
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Try adding timezone ONLY if needed
try:
    df['date'] = (
        df['date']
        .dt.tz_localize('Etc/GMT+4', ambiguous='NaT', nonexistent='NaT')
        .dt.tz_convert('UTC')
    )
except:
    pass

# Show missing values
print("Missing headlines:", df['headline'].isna().sum())
print("Missing stock tickers:", df['stock'].isna().sum())

df[['headline','stock','publisher','date']].head(10)


In [ ]:
# Headline length
df['headline_len'] = df['headline'].astype(str).str.len()
df['headline_len'].describe()


In [ ]:
# Top publishers
pub_counts = df['publisher'].fillna('UNKNOWN').value_counts().head(30)
pub_counts


In [ ]:
df_cal = df.copy()

if hasattr(df_cal['date'].dt, 'tz'):
    df_cal['day'] = df_cal['date'].dt.tz_convert('UTC').dt.floor('D')
else:
    df_cal['day'] = df_cal['date'].dt.floor('D')

daily_counts = df_cal.groupby('day').size().rename('article_count')
daily_counts.sort_values(ascending=False).head(10)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.hist(df['headline_len'].dropna(), bins=40)
plt.title("Headline Length Distribution")
plt.xlabel("Characters")
plt.ylabel("Count")
plt.show()


In [ ]:
daily_counts_tail = daily_counts.sort_index().tail(180)

plt.figure(figsize=(12,3))
plt.plot(daily_counts_tail.index, daily_counts_tail.values)
plt.title("Daily Article Counts (Last 180 Days)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer(stop_words='english', ngram_range=(1,2), max_features=2000)
X = vect.fit_transform(df['headline'].fillna(''))

sums = X.sum(axis=0)
terms = [(term, sums[0, idx]) for term, idx in vect.vocabulary_.items()]

top = sorted(terms, key=lambda x: x[1], reverse=True)[:30]
print("Top tokens:")
for t, c in top:
    print(f"{t}: {c}")


In [ ]:
publisher_counts = (
    df['publisher']
    .fillna('UNKNOWN')
    .value_counts()
    .reset_index()
)

publisher_counts.columns = ['publisher', 'count']

publisher_counts.to_csv(
    '/content/drive/MyDrive/Data-20251119T090255Z-1-001-week1/publisher_counts.csv',
    index=False
)

print("Saved publisher_counts.csv ✔")
